###**Programación Concurrente**
####Actividad Práctica (Opcional) - Procesos Pesados

---



---

##**Ejercicio 2 - *Python***




Para compilar y ejecutar codigo en Python en Google Colab:

1.  **Python ya está instalado**: Google Colab incluye Python por defecto, por lo que no es necesario instalarlo.
2.  **Escribir el código Python**: El código se puede escribir directamente en una celda de código de Google Colab
3.  **Compilar el codigo Java**: Para ejecutar el programa, presiona el botón ▶️ de la celda o utiliza Shift + Enter

In [ ]:
%%writefile jurassic_park_monitor.py
import threading
import random
import time
import sys

EXPECTED_ARGUMENT_COUNT = 3
MIN_ALLOWED_VALUE = 0
SINGLE_EVENT_COUNT = 1
FIRST_ITEM_INDEX = 0
FIRST_ARGUMENT_INDEX = 1
SECOND_ARGUMENT_INDEX = 2

# Eventos para cada zona y sus probabilidades
ZONAS = {
    "Velociraptor Area": [
        ("All clear", 70),
        ("Loss of visibility", 20),
        ("Electric fence failure", 10)
    ],

    "Tyrannosaurus Sector": [
        ("All clear", 80),
        ("Tyrannosaurus outside the enclosure", 10),
        ("Electric fence failure", 10)
    ],

    "Triceratops Enclosure": [
        ("All clear", 60),
        ("Unusual behavior", 30),
        ("Stampede", 10)
    ],

    "Visitor Center": [
        ("All clear", 80),
        ("Loss of communication", 15),
        ("Security alert", 5)
    ],

    "Genetic Laboratory": [
        ("All clear", 80),
        ("System failure", 10),
        ("Loss of communication", 5),
        ("Unauthorized access", 5)
    ]
}


# Eventos considerados críticos
CRITICAL_EVENTS = {
    "Tyrannosaurus outside the enclosure",
    "Electric fence failure",
    "Loss of communication",
    "Security alert"
}


# Bloqueo para evitar que dos hilos impriman al mismo tiempo
print_lock = threading.Lock()


def monitor_zone(zone, events, duration, frequency):
    total_events = 0
    total_critical = 0

    start_time = time.monotonic()

    while time.monotonic() - start_time < duration:

        # Seleccionar un evento según su probabilidad
        event_names = [event[0] for event in events]
        probabilities = [event[1] for event in events]

        event = random.choices(
            event_names,
            weights=probabilities,
            k=SINGLE_EVENT_COUNT
        )[FIRST_ITEM_INDEX]

        total_events += 1

        if event in CRITICAL_EVENTS:
            total_critical += 1

        # Imprimir el evento detectado
        with print_lock:
            print(f"[{zone}] - {event}")

        # Esperar hasta el próximo informe
        elapsed_time = time.monotonic() - start_time
        remaining_time = duration - elapsed_time

        if remaining_time <= MIN_ALLOWED_VALUE:
            break

        time.sleep(min(frequency, remaining_time))

    # Informe final del sistema
    with print_lock:
        print()
        print(f"--- Final report: {zone} ---")
        print(f"Total events detected: {total_events}")
        print(f"Total critical events: {total_critical}")
        print()


def main():

    # Validar parámetros
    if len(sys.argv) != EXPECTED_ARGUMENT_COUNT:
        print("Usage: python jurassic_park_monitor.py <duration> <frequency>")
        print("Example: python jurassic_park_monitor.py 20 3")
        return

    duration = float(sys.argv[FIRST_ARGUMENT_INDEX])
    frequency = float(sys.argv[SECOND_ARGUMENT_INDEX])

    if (duration <= MIN_ALLOWED_VALUE
            or frequency <= MIN_ALLOWED_VALUE):
        print("Duration and frequency must be greater than 0.")
        return

    threads = []

    # Crear un hilo para cada zona
    for zone, events in ZONAS.items():

        thread = threading.Thread(
            target=monitor_zone,
            args=(zone, events, duration, frequency)
        )

        threads.append(thread)
        thread.start()

    # Esperar a que todos los sistemas terminen
    for thread in threads:
        thread.join()

    print("===================================")
    print("Park monitoring completed.")
    print("===================================")


if __name__ == "__main__":
    main()

Writing jurassic_park_monitor.py


Para ejecutar el programa:

In [ ]:
!python jurassic_park_monitor.py 20 3

[Velociraptor Area] - All clear
[Tyrannosaurus Sector] - All clear
[Triceratops Enclosure] - Stampede
[Visitor Center] - All clear
[Genetic Laboratory] - All clear
[Velociraptor Area] - Loss of visibility
[Tyrannosaurus Sector] - All clear
[Triceratops Enclosure] - Unusual behavior
[Visitor Center] - All clear
[Genetic Laboratory] - All clear
[Velociraptor Area] - All clear
[Tyrannosaurus Sector] - All clear
[Triceratops Enclosure] - All clear
[Visitor Center] - All clear
[Genetic Laboratory] - All clear
[Velociraptor Area] - All clear
[Triceratops Enclosure] - Stampede
[Tyrannosaurus Sector] - All clear
[Visitor Center] - All clear
[Genetic Laboratory] - All clear
[Velociraptor Area] - All clear
[Tyrannosaurus Sector] - All clear
[Genetic Laboratory] - All clear
[Triceratops Enclosure] - Unusual behavior
[Visitor Center] - All clear
[Velociraptor Area] - Electric fence failure
[Genetic Laboratory] - All clear
[Visitor Center] - Security alert
[Triceratops Enclosure] - All clear
[Tyran

**Concluciones**

Con esta actividad comprobamos la creación y el ciclo de vida de procesos concurrentes. Al ejecutar múltiples tareas de monitoreo en simultáneo, observamos el no determinismo inherente a la concurrencia, reflejado en el orden impredecible de llegada de los eventos. Dado que todas las instancias competían por la consola como recurso compartido, fue necesario delimitar la región crítica mediante un lock. Aunque la concurrencia hace al sistema mucho más eficiente, cuando varias tareas comparten un mismo recurso es obligatorio sincronizarlas para evitar errores.